In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
import os
import argparse
from pathlib import Path
import re
import random
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Model and Loss

In [ ]:

import math
import numpy as np
import torch
from torch import nn

class ChannelAwareNAOMI(nn.Module):
    def __init__(self, data_dim=2, hidden_dim=64, decoder_hidden=128, highest=16,
                 rnn_layers=1, dropout=0.0, normalize=True):
        super().__init__()
        assert highest > 0 and highest & (highest - 1) == 0
        self.data_dim = data_dim
        self.hidden_dim = hidden_dim
        self.highest = highest
        self.normalize = normalize
        self.encoder = nn.GRU(data_dim*2, hidden_dim, num_layers=rnn_layers,
                              bidirectional=True, batch_first=True,
                              dropout=dropout if rnn_layers > 1 else 0.)
        self.decoders = nn.ModuleDict()
        for step in (2**p for p in range(int(math.log2(highest))+1)):
            self.decoders[str(step)] = nn.Sequential(
                nn.Linear(hidden_dim*2 + data_dim*2 + 2, decoder_hidden),
                nn.ReLU(), nn.Linear(decoder_hidden, data_dim))
        # CPU schedule cache, not checkpoint parameters. One entry per unique mask.
        self._schedule_cache = {}

    def _norm(self, x, m):
        valid = m.bool()
        count = valid.sum(dim=1, keepdim=True)
        safe_count = count.clamp_min(1)
        observed = torch.where(valid, x, torch.zeros_like(x))
        mean = observed.sum(dim=1, keepdim=True) / safe_count
        variance = torch.where(valid, (x-mean).square(), torch.zeros_like(x)).sum(dim=1, keepdim=True) / safe_count
        sd = (variance + 1e-6).sqrt()
        empty = count == 0
        mean = torch.where(empty, torch.zeros_like(mean), mean)
        sd = torch.where(empty, torch.ones_like(sd), sd)
        if not self.normalize:
            mean = torch.zeros_like(mean)
            sd = torch.ones_like(sd)
        normalized = torch.where(valid, (x-mean)/sd, torch.zeros_like(x))
        return normalized, mean, sd

    def _make_schedule(self, mask):
        """Generate original midpoint pivots from an unchanging (T,D) CPU mask."""
        T,D = mask.shape
        available = mask.copy()
        rounds = []
        max_rounds = int(math.ceil(math.log2(T)))+2
        for _ in range(max_rounds):
            if available.all():
                break
            entries = []
            for d in range(D):
                missing = np.flatnonzero(~available[:,d])
                if missing.size == 0:
                    continue
                splits = np.flatnonzero(np.diff(missing) != 1)+1
                for run in np.split(missing, splits):
                    a,z = int(run[0]),int(run[-1])
                    if a == 0 and z == T-1:
                        l,r = 0,T-1
                    else:
                        l,r = max(0,a-1),min(T-1,z+1)
                        if a == 0: l=r
                        if z == T-1: r=l
                    pivot=(a+z)//2
                    dist=max(1,r-l)
                    step=min(self.highest, 2**min(int(math.log2(dist)),int(math.log2(self.highest))))
                    entries.append((pivot,d,l,r,step,dist/T,(pivot-l)/max(1,dist)))
                    available[pivot,d] = True
            if not entries:
                raise RuntimeError('No missing pivots generated')
            rounds.append(entries)
        if not available.all():
            raise RuntimeError('Failed to fill all gaps')
        return rounds

    def forward(self, x_in, m_hat):
        if x_in.ndim != 3 or x_in.shape != m_hat.shape or x_in.shape[-1] != self.data_dim:
            raise ValueError('Inputs must be (B,T,D) and have matching masks')
        if not torch.isfinite(x_in).all().item():
            raise ValueError('X_in must be finite and prefilled')
        x,mean,sd = self._norm(x_in,m_hat)
        B,T,D = x.shape
        device = x.device
        mask_np = m_hat.detach().to('cpu').numpy() > 0.5   # once per forward; no per-pivot GPU sync
        per_sample = []
        for b in range(B):
            key = (T,D, np.packbits(mask_np[b].reshape(-1)).tobytes())
            schedule = self._schedule_cache.get(key)
            if schedule is None:
                schedule = self._make_schedule(mask_np[b])
                self._schedule_cache[key] = schedule
            per_sample.append(schedule)
        max_rounds = max((len(s) for s in per_sample),default=0)
        work = x
        available = m_hat.bool().clone()
        for round_i in range(max_rounds):
            recurrent,_ = self.encoder(torch.cat([work, available.to(work.dtype)],dim=-1))
            fwd = recurrent[:,:,:self.hidden_dim]
            bwd = recurrent[:,:,self.hidden_dim:]
            groups = {step:[] for step in (2**p for p in range(int(math.log2(self.highest))+1))}
            for b,schedule in enumerate(per_sample):
                if round_i >= len(schedule):
                    continue
                for t,d,l,r,step,dist_fraction,pivot_fraction in schedule[round_i]:
                    groups[step].append((b,t,d,l,r,dist_fraction,pivot_fraction))
            positions_list = []
            values_list = []
            for step, entries in groups.items():
                if not entries:
                    continue
                info_np = np.asarray(entries,dtype=np.float64)
                b = torch.as_tensor(info_np[:,0],device=device,dtype=torch.long)
                t = torch.as_tensor(info_np[:,1],device=device,dtype=torch.long)
                d = torch.as_tensor(info_np[:,2],device=device,dtype=torch.long)
                l = torch.as_tensor(info_np[:,3],device=device,dtype=torch.long)
                r = torch.as_tensor(info_np[:,4],device=device,dtype=torch.long)
                positions = b*(T*D)+t*D+d
                features = torch.cat((fwd[b,l],bwd[b,r],work[b,l],work[b,r],
                                      torch.as_tensor(info_np[:,5:7],device=device,dtype=work.dtype)),dim=-1)
                values = self.decoders[str(step)](features).gather(1,d[:,None]).squeeze(1)
                positions_list.append(positions)
                values_list.append(values)
            if not positions_list:
                raise RuntimeError('No missing pivots generated')
            positions=torch.cat(positions_list)
            values=torch.cat(values_list)
            work=work.reshape(-1).scatter(0,positions,values).view(B,T,D)
            available=available.reshape(-1).scatter(0,positions,
                        torch.ones_like(positions,dtype=torch.bool)).view(B,T,D)
        x_hat=work*sd+mean
        x_c=torch.where(m_hat.bool(),x_in,x_hat)
        return {'x_hat':x_hat,'x_c':x_c,'x_norm_hat':work,'mu':mean,'sigma':sd}


def masked_target_loss(outputs,truth,observed,holdout,loss_type='mse'):
    mask=holdout.bool() & (~observed.bool()) & torch.isfinite(truth)
    if not mask.any().item():
        return None
    errors=outputs['x_c'][mask]-truth[mask]
    if loss_type=='mse': return errors.square().mean()
    if loss_type=='mae': return errors.abs().mean()
    raise ValueError(loss_type)


### Model training

In [ ]:
NPZ_PATH = '/content/drive/MyDrive/aim3/saits_masks_1800x2_reintroduced_exact_gaps_capped360_with_bio_numeric.npz'
OUT_ROOT = '/content/drive/MyDrive/aim3/NAOMI_SingleChannel_5fold_1e-3_64'

In [ ]:
"""Five-fold patient-level SINGLE-CHANNEL ablation for channel-aware NAOMI.

This script trains two INDEPENDENT configurations (choose both or either):
  * ScO2-only: original channel 0 -> model shape (B, 1080, 1)
  * MAP-only:  original channel 1 -> model shape (B, 1080, 1)

It DOES NOT modify the NAOMI model architecture or its masked-target MSE loss.
The model remains a channel-aware NAOMI ADAPTATION, not official NAOMI.

Place the original model file `naomi_channel_aware.py` or the faster compatible
`naomi_channel_aware_fast.py` beside this script. Then in Colab:

    %run /content/drive/MyDrive/aim3/train_naomi_single_channel_5fold.py

Original (B, 1080, 2) NPZ arrays, segment IDs, five-second gap definitions,
patient splitting, 300-epoch cap, batch size 64, early stopping and validation
MAE checkpoint selection are preserved from the user's dual-channel trainer.

Separate output folders prevent overwriting existing dual-channel checkpoints.
"""

import importlib.util
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# ============================== CONFIG =====================================

DUAL_SPLIT_CSV = '/content/drive/MyDrive/aim3/NAOMI_ChannelAware_5fold_1e-3_64/cv5_patient_split_summary.csv'

CHANNELS_TO_TRAIN = ('ScO2', 'MAP')
ORIGINAL_CHANNELS = {'ScO2': 0, 'MAP': 1}

SEED = 1337
N_SPLITS = 5
VAL_WITHIN_TRAIN_P = 0.10
BATCH_SIZE = 64
EPOCHS = 300
LR = 1e-3
WEIGHT_DECAY = 0.0
EARLY_STOPPING_PATIENCE = 15
CLIP_GRAD_NORM = 1.0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE_MODEL_KWARGS = dict(hidden_dim=64, decoder_hidden=128,
                         highest=16, rnn_layers=1, normalize=True)
REGIONS = ['<1 min', '[1,10) min', '[10,20) min', '[20,30] min']
EPS = 1e-8


def import_naomi():
    """Load user's UNCHANGED model file; no architecture is reimplemented."""
    script_dir = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
    candidates = []
    for directory in (script_dir, Path.cwd(), Path('/content/drive/MyDrive/aim3')):
        for name in ('naomi_channel_aware_fast.py', 'naomi_channel_aware.py'):
            path = directory / name
            if path not in candidates:
                candidates.append(path)
    for path in candidates:
        if path.is_file():
            spec = importlib.util.spec_from_file_location('_naomi_ablation_model', path)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            if not hasattr(mod, 'ChannelAwareNAOMI') or not hasattr(mod, 'masked_target_loss'):
                raise ImportError(f'{path} must define ChannelAwareNAOMI and masked_target_loss')
            print('NAOMI model loaded from:', path)
            return mod.ChannelAwareNAOMI, mod.masked_target_loss
    if 'ChannelAwareNAOMI' in globals() and 'masked_target_loss' in globals():
        print('Using pre-defined ChannelAwareNAOMI and masked_target_loss')
        return globals()['ChannelAwareNAOMI'], globals()['masked_target_loss']
    raise FileNotFoundError('Place naomi_channel_aware_fast.py or naomi_channel_aware.py '
                            'in the same directory as this training script.')


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def save(df, directory, filename):
    os.makedirs(directory, exist_ok=True)
    path = os.path.join(directory, filename)
    df.to_csv(path, index=False)
    return path


class DatasetImpute(Dataset):
    def __init__(self, arrays, indices):
        x, m, y, i = arrays
        self.x = torch.from_numpy(np.nan_to_num(x[indices], nan=0., posinf=0., neginf=0.).astype('float32'))
        self.m = torch.from_numpy(np.nan_to_num(m[indices], nan=0., posinf=0., neginf=0.).astype('float32'))
        # Preserve original GT NaN: never score naturally unavailable targets.
        self.y = torch.from_numpy(y[indices].astype('float32'))
        self.i = torch.from_numpy(np.nan_to_num(i[indices], nan=0., posinf=0., neginf=0.).astype('float32'))
        self.ids = indices

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, j):
        return self.x[j], self.m[j], self.y[j], self.i[j], int(self.ids[j])


def make_loader(arrays, ids, shuffle=False):
    return DataLoader(DatasetImpute(arrays, ids), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=0, pin_memory=DEVICE == 'cuda')


def to_device(batch):
    x, m, y, i, indices = batch
    return x.to(DEVICE), m.to(DEVICE), y.to(DEVICE), i.to(DEVICE), indices


def gap_lengths(mask):
    m = np.asarray(mask) > 0.5
    edges = np.diff(np.r_[False, m, False].astype('int8'))
    result = np.zeros(len(m), dtype='int32')
    for start, stop in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)):
        result[start:stop] = stop - start
    return result


def region(n):
    if 1 <= n < 12: return REGIONS[0]
    if 12 <= n < 120: return REGIONS[1]
    if 120 <= n < 240: return REGIONS[2]
    if 240 <= n <= 360: return REGIONS[3]
    return None


def metrics(frame):
    yt = frame['y_true'].to_numpy(float)
    yp = frame['y_pred'].to_numpy(float)
    good = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[good], yp[good]
    if len(yt) == 0:
        return dict(mae=np.nan, rmse=np.nan, mape_pct=np.nan, n_points=0)
    err = np.abs(yp - yt)
    positive = np.abs(yt) > EPS
    return dict(mae=float(err.mean()), rmse=float(np.sqrt(np.mean(err**2))),
                mape_pct=float(100 * np.mean(err[positive] / np.abs(yt[positive])))
                         if positive.any() else np.nan,
                n_points=int(len(yt)))


def run_train_epoch(model, loader, optimizer, masked_target_loss):
    model.train()
    total, count = 0., 0
    for batch in loader:
        x, m, y, i, _ = to_device(batch)
        optimizer.zero_grad(set_to_none=True)
        output = model(x, m)
        loss = masked_target_loss(output, y, m, i, loss_type='mse')
        if loss is None:
            continue
        if not torch.isfinite(loss).item():
            raise FloatingPointError('Nonfinite single-channel NAOMI training loss')
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD_NORM)
        optimizer.step()
        n = int((i.bool() & ~m.bool() & torch.isfinite(y)).sum())
        total += float(loss.detach()) * n
        count += n
    if not count:
        raise ValueError('No valid artificial targets in training split for this channel')
    return total / count


@torch.no_grad()
def run_eval(model, loader):
    model.eval()
    abs_sum, sq_sum, n = 0., 0., 0
    for batch in loader:
        x, m, y, i, _ = to_device(batch)
        output = model(x, m)
        mask = i.bool() & (~m.bool()) & torch.isfinite(y)
        errors = output['x_c'][mask] - y[mask]
        if errors.numel():
            abs_sum += float(errors.abs().sum())
            sq_sum += float(errors.square().sum())
            n += errors.numel()
    if not n:
        raise ValueError('No valid artificial targets in evaluation split for this channel')
    return dict(mae=abs_sum / n, mse=sq_sum / n,
                rmse=(sq_sum / n) ** .5, n_points=n)


@torch.no_grad()
def collect_predictions(model, loader, patient_ids, filenames, fold, split, channel_name, model_name):
    """CRITICAL: local model channel 0 is ScO2 OR MAP, per channel_name."""
    model.eval()
    rows = []
    for batch in loader:
        x, m, y, i, indices = to_device(batch)
        pred = model(x, m)['x_c'].cpu().numpy()
        truth = y.cpu().numpy()
        target = i.cpu().numpy()
        visible = m.cpu().numpy()
        assert pred.shape[-1] == truth.shape[-1] == 1
        for b, idx in enumerate(indices.tolist()):
            d = 0  
            lengths = gap_lengths(target[b, :, d])
            scored = np.flatnonzero((target[b, :, d] > .5) & (visible[b, :, d] <= .5))
            for t in scored:
                yt, yp = float(truth[b, t, d]), float(pred[b, t, d])
                if not (np.isfinite(yt) and np.isfinite(yp)):
                    continue
                gap_len = int(lengths[t])
                rows.append(dict(
                    fold=fold, split=split, segment_idx=int(idx), patient_id=patient_ids[idx],
                    filename=str(filenames[idx]), time_idx=int(t), target=channel_name,
                    model=model_name, y_true=yt, y_pred=yp, abs_err=abs(yp-yt),
                    gap_len=gap_len, gap_region=region(gap_len)
                ))
    return pd.DataFrame(rows)


def grouped_metrics(frame, keys):
    if frame.empty:
        return pd.DataFrame()
    result = []
    for key, group in frame.groupby(keys, observed=True):
        if not isinstance(key, tuple):
            key = (key,)
        result.append({**dict(zip(keys, key)), **metrics(group)})
    return pd.DataFrame(result)


def save_summaries(out_dir, split_rows, fold_rows, test_frames, val_frames):
    folds = pd.DataFrame(fold_rows)
    tests = pd.concat(test_frames, ignore_index=True)
    vals = pd.concat(val_frames, ignore_index=True)
    save(pd.DataFrame(split_rows), out_dir, 'cv5_patient_split_summary.csv')
    save(folds, out_dir, 'cv5_fold_results.csv')
    save(tests, out_dir, 'cv5_test_predictions_pooled.csv')
    save(vals, out_dir, 'cv5_validation_predictions_pooled.csv')
    for split, frame in [('test', tests), ('validation', vals)]:
        gap = frame.loc[frame['gap_region'].notna()].copy()
        save(gap, out_dir, f'cv5_{split}_predictions_pooled_with_gap_region.csv')
        save(grouped_metrics(frame, ['target', 'model']), out_dir,
             f'cv5_pooled_{split}_metrics.csv')
        save(grouped_metrics(gap, ['target', 'model', 'gap_region']), out_dir,
             f'cv5_pooled_{split}_metrics_by_gap_region.csv')
        fg = grouped_metrics(gap, ['fold', 'target', 'model', 'gap_region'])
        save(fg, out_dir, f'cv5_{split}_fold_metrics_by_gap_region.csv')
        if split == 'test':
            save(fg, out_dir, 'cv5_fold_metrics_by_gap_region.csv')
        if not fg.empty:
            summary = fg.groupby(['target', 'model', 'gap_region'], observed=True, as_index=False).agg(
                mae_mean=('mae', 'mean'), mae_std=('mae', 'std'),
                rmse_mean=('rmse', 'mean'), rmse_std=('rmse', 'std'),
                mape_mean_pct=('mape_pct', 'mean'), mape_std_pct=('mape_pct', 'std'),
                n_folds=('fold', 'nunique'), total_points=('n_points', 'sum'))
            save(summary, out_dir, f'cv5_{split}_summary_mean_std_by_gap_region.csv')
            if split == 'test':
                save(summary, out_dir, 'cv5_summary_mean_std_by_gap_region.csv')
    overall = folds.groupby(['target', 'model'], as_index=False).agg(
        val_mae_mean=('val_mae', 'mean'), val_mae_std=('val_mae', 'std'),
        test_mae_mean=('test_mae', 'mean'), test_mae_std=('test_mae', 'std'),
        test_rmse_mean=('test_rmse', 'mean'), test_rmse_std=('test_rmse', 'std'),
        test_mape_mean_pct=('test_mape_pct', 'mean'), test_mape_std_pct=('test_mape_pct', 'std'),
        n_folds=('fold', 'nunique'), total_points=('n_points', 'sum'))
    save(overall, out_dir, 'cv5_summary_mean_std.csv')
    print('\nSingle-channel fold summary:\n', overall.to_string(index=False))
    return overall


def build_patient_splits(seg_id):
    """Exactly preserve the random split algorithm from the dual-channel script."""
    patients = np.unique(seg_id)
    shuffled = patients.copy()
    np.random.default_rng(SEED).shuffle(shuffled)
    test_folds = np.array_split(shuffled, N_SPLITS)
    split_rows = []
    for fold, test_pats in enumerate(test_folds, 1):
        rest = patients[~np.isin(patients, test_pats)].copy()
        np.random.default_rng(SEED + fold).shuffle(rest)
        nv = min(max(1, round(len(rest) * VAL_WITHIN_TRAIN_P)), len(rest) - 1)
        val_pats, train_pats = rest[:nv], rest[nv:]
        tr = np.flatnonzero(np.isin(seg_id, train_pats))
        va = np.flatnonzero(np.isin(seg_id, val_pats))
        te = np.flatnonzero(np.isin(seg_id, test_pats))
        if not (len(tr) and len(va) and len(te)):
            raise ValueError(f'Fold {fold} has an empty train, validation, or test segment split')
        if set(train_pats) & set(val_pats) or set(train_pats) & set(test_pats) or set(val_pats) & set(test_pats):
            raise ValueError(f'Fold {fold}: patient leakage')
        row = dict(fold=fold,
                   train_patients=','.join(map(str, train_pats)),
                   val_patients=','.join(map(str, val_pats)),
                   test_patients=','.join(map(str, test_pats)),
                   n_train_patients=len(train_pats), n_val_patients=len(val_pats),
                   n_test_patients=len(test_pats),
                   n_train_segments=len(tr), n_val_segments=len(va), n_test_segments=len(te))
        split_rows.append((row, tr, va, te))
    return split_rows


def check_existing_dual_splits(split_rows):
    if not os.path.isfile(DUAL_SPLIT_CSV):
        print('Dual-channel split CSV not found; splits are reproduced using original seed/code:', SEED)
        return
    original = pd.read_csv(DUAL_SPLIT_CSV)
    if 'fold' not in original.columns:
        raise ValueError(f'Dual-channel split CSV lacks fold column: {DUAL_SPLIT_CSV}')
    if original['fold'].duplicated().any():
        raise ValueError('Duplicate fold entries in dual-channel split CSV')
    for row, _, _, _ in split_rows:
        old = original.loc[original['fold'] == row['fold']]
        if old.empty:
            continue  # Original dual training may still be running.
        old = old.iloc[0]
        for name in ('train_patients', 'val_patients', 'test_patients'):
            parse = lambda s: {int(float(v.strip())) for v in str(s).split(',') if v.strip()}
            if parse(old[name]) != parse(row[name]):
                raise ValueError(f'Fold {row["fold"]}: {name} differs from existing dual-channel split. '
                                 'Do not compare unmatched patients.')
    print('Patient partitions verified against:', DUAL_SPLIT_CSV)


def train_channel(channel_name, channel_index, data, splits, model_class, loss_function):
    x, m, y, i, seg_id, files = data
    output_dir = os.path.join(OUT_ROOT, f'{channel_name}_only')
    ckpt_dir = os.path.join(output_dir, 'ckpts')
    os.makedirs(ckpt_dir, exist_ok=True)
    model_name = f'NAOMI_ChannelAware_Adapted_{channel_name}_only'
    # Slice EACH array identically. Use a slice to preserve the last dimension.
    sub = slice(channel_index, channel_index + 1)
    arrays = (x[:, :, sub], m[:, :, sub], y[:, :, sub], i[:, :, sub])
    assert all(a.shape == (len(x), 1080, 1) for a in arrays)
    split_rows, fold_rows, test_frames, val_frames = [], [], [], []
    print(f'\n{"=" * 75}\n{channel_name}-ONLY | original channel {channel_index} | '
          f'{arrays[0].shape} | output: {output_dir}\n{"=" * 75}')
    for row, tr, va, te in splits:
        fold = row['fold']
        print(f'\n{channel_name} fold {fold}/{N_SPLITS}: '
              f'train/val/test segments {len(tr)}/{len(va)}/{len(te)}')
        split_rows.append(row)
        save(pd.DataFrame(split_rows), output_dir, 'cv5_patient_split_summary.csv')
        train_loader = make_loader(arrays, tr, True)
        val_loader = make_loader(arrays, va)
        test_loader = make_loader(arrays, te)
        seed_all(SEED + fold)
        model = model_class(data_dim=1, **BASE_MODEL_KWARGS).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=3)
        ckpt = os.path.join(ckpt_dir, f'fold{fold:02d}_best.pt')
        best, best_epoch, no_improve, history = float('inf'), None, 0, []
        for epoch in range(1, EPOCHS + 1):
            train_mse = run_train_epoch(model, train_loader, optimizer, loss_function)
            val = run_eval(model, val_loader)
            scheduler.step(val['mae'])
            history.append(dict(fold=fold, epoch=epoch,
                                learning_rate=optimizer.param_groups[0]['lr'],
                                train_masked_mse=train_mse,
                                val_masked_mse=val['mse'], val_masked_mae=val['mae'],
                                val_masked_rmse=val['rmse']))
            save(pd.DataFrame(history), output_dir, f'fold{fold:02d}_training_history.csv')
            print(f'  Epoch {epoch:03d} lr={optimizer.param_groups[0]["lr"]:.2e} '
                  f'train MSE={train_mse:.5f} | val MAE={val["mae"]:.5f} '
                  f'RMSE={val["rmse"]:.5f}')
            if val['mae'] < best:
                best, best_epoch, no_improve = val['mae'], epoch, 0
                torch.save(model.state_dict(), ckpt)  # state_dict, as original
            else:
                no_improve += 1
                if no_improve >= EARLY_STOPPING_PATIENCE:
                    print('  Early stop at', epoch)
                    break
        if best_epoch is None:
            raise ValueError(f'No checkpoint selected for {channel_name} fold {fold}')
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
        vm = run_eval(model, val_loader)
        vp = collect_predictions(model, val_loader, seg_id, files, fold, 'val',
                                 channel_name, model_name)
        tp = collect_predictions(model, test_loader, seg_id, files, fold, 'test',
                                 channel_name, model_name)
        if vp.empty or tp.empty:
            raise ValueError(f'Empty {channel_name} predictions in fold {fold}')
        save(vp, output_dir, f'fold{fold:02d}_validation_predictions.csv')
        save(tp, output_dir, f'fold{fold:02d}_test_predictions.csv')
        val_frames.append(vp)
        test_frames.append(tp)
        mv, mt = metrics(vp), metrics(tp)
        fold_rows.append(dict(fold=fold, target=channel_name, model=model_name,
                              best_epoch=best_epoch, val_mae=mv['mae'],
                              val_overall_mae=vm['mae'], test_mae=mt['mae'],
                              test_rmse=mt['rmse'], test_mape_pct=mt['mape_pct'],
                              n_points=mt['n_points']))
        print(f'  TEST {channel_name}: MAE={mt["mae"]:.4f} '
              f'RMSE={mt["rmse"]:.4f} MAPE={mt["mape_pct"]:.2f}%')
        save_summaries(output_dir, split_rows, fold_rows, test_frames, val_frames)
        del model, optimizer, scheduler
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
    print(f'Completed {channel_name}-only 5-fold. Files saved in {output_dir}')
    return pd.read_csv(os.path.join(output_dir, 'cv5_summary_mean_std.csv'))


def main():
    seed_all(SEED)
    if not CHANNELS_TO_TRAIN or any(ch not in ORIGINAL_CHANNELS for ch in CHANNELS_TO_TRAIN):
        raise ValueError('CHANNELS_TO_TRAIN must contain "ScO2" and/or "MAP"')
    model_class, loss_function = import_naomi()
    with np.load(NPZ_PATH, allow_pickle=True) as data:
        x = np.asarray(data['X_in'])
        m = np.asarray(data['M_hat'])
        y = np.asarray(data['X'])
        i = np.asarray(data['I'])
        seg_id = np.asarray(data['segment_ids'])
        files = (np.asarray(data['filenames']).astype(str) if 'filenames' in data.files
                 else np.array([f'segment_{k}' for k in range(len(x))]))
        original_labels = (list(map(str, data['target_columns']))
                           if 'target_columns' in data.files else None)
    if not (x.shape == m.shape == y.shape == i.shape) or x.shape[1:] != (1080, 2):
        raise ValueError(f'Expected aligned (N, 1080, 2) arrays, got {x.shape}')
    if len(seg_id) != len(x):
        raise ValueError('segment_ids and segment count differ')
    if np.any((m > .5) & (i > .5)):
        raise ValueError('Observed M_hat and artificial I overlap')
    if not np.isfinite(y[i > .5]).all():
        raise ValueError('Artificial target ground truth contains nonfinite values')
    if not np.isfinite(x[m > .5]).all():
        raise ValueError('Observed input contains nonfinite values')
    if original_labels is not None:
        print('NPZ target_columns:', original_labels)
        canonical = [str(v).lower().replace(' ', '').replace('_', '').replace('₂', '2')
                     for v in original_labels]
        if len(canonical) >= 2 and canonical[:2] == ['map', 'sco2']:
            raise ValueError('NPZ target_columns implies channel order MAP, ScO2, '
                             'contrary to the assumed channel 0 ScO2 / channel 1 MAP')
    print('Original shape:', x.shape, '| patients:', len(np.unique(seg_id)), '| device:', DEVICE)
    print('Original channel map: channel 0 = ScO2, channel 1 = MAP')
    print('Loss: masked artificial-target MSE; selection: validation MAE')
    splits = build_patient_splits(seg_id)
    check_existing_dual_splits(splits)
    os.makedirs(OUT_ROOT, exist_ok=True)
    summaries = []
    for channel in CHANNELS_TO_TRAIN:
        result = train_channel(channel, ORIGINAL_CHANNELS[channel],
                               (x, m, y, i, seg_id, files), splits,
                               model_class, loss_function)
        summaries.append(result)
    if summaries:
        combined = pd.concat(summaries, ignore_index=True)
        print('\nFINAL SINGLE-CHANNEL ABLATION RESULTS:\n', combined.to_string(index=False))
        save(combined, OUT_ROOT, 'cv5_single_channel_summary_mean_std.csv')
    print('All single-channel results saved under:', OUT_ROOT)


if __name__ == '__main__':
    main()


Using pre-defined ChannelAwareNAOMI and masked_target_loss
Original shape: (3817, 1080, 2) | patients: 64 | device: cuda
Original channel map: channel 0 = ScO2, channel 1 = MAP
Loss: masked artificial-target MSE; selection: validation MAE
Patient partitions verified against: /content/drive/MyDrive/aim3/NAOMI_ChannelAware_5fold_1e-3_64/cv5_patient_split_summary.csv

ScO2-ONLY | original channel 0 | (3817, 1080, 1) | output: /content/drive/MyDrive/aim3/NAOMI_SingleChannel_5fold_1e-3_64/ScO2_only

ScO2 fold 1/5: train/val/test segments 2814/256/747
  Epoch 001 lr=1.00e-03 train MSE=8.36858 | val MAE=1.63837 RMSE=2.73602
  Epoch 002 lr=1.00e-03 train MSE=5.34997 | val MAE=1.98043 RMSE=3.18678
  Epoch 003 lr=1.00e-03 train MSE=4.87993 | val MAE=1.59655 RMSE=2.64741
  Epoch 004 lr=1.00e-03 train MSE=4.19383 | val MAE=1.52959 RMSE=2.55433
  Epoch 005 lr=1.00e-03 train MSE=3.94095 | val MAE=1.54355 RMSE=2.54597
  Epoch 006 lr=1.00e-03 train MSE=3.99612 | val MAE=1.70837 RMSE=2.76532
  Epoch 00

### Model Testing

In [ ]:
"""Five-fold held-out TEST MAE/MAPE reporting for SINGLE-CHANNEL NAOMI.

Run after train_naomi_single_channel_5fold.py has completed all five folds.
No training, model imports, or checkpoint loading.

Reads:
  <OUT_ROOT>/ScO2_only/cv5_test_predictions_pooled.csv
  <OUT_ROOT>/MAP_only/cv5_test_predictions_pooled.csv

Original channel 0 = ScO2, original channel 1 = MAP.  The single-channel
training code writes explicit 'ScO2'/'MAP' target labels even though its
sliced local channel index is 0 for BOTH models.

Artificial gap bins at 5-second samples:
  1..11 (<1 min), 12..119 ([1,10) min), 120..239 ([10,20) min),
  240..360 ([20,30] min). ALL includes every finite saved held-out target.
Five-fold mean +/- sample SD treats each fold equally; pooled scores give
all points equal weight. Missing gap bins can produce n_folds < 5 and SD=NaN.

Optionally verifies and compares paired predictions with existing dual-channel
NAOMI. The comparison is performed only when the dual-channel test CSV exists.
"""

from pathlib import Path
import numpy as np
import pandas as pd

# =============================== CONFIG ====================================
OUT_ROOT = Path('/content/drive/MyDrive/aim3/NAOMI_SingleChannel_5fold_1e-3_64')
DUAL_DIR = Path('/content/drive/MyDrive/aim3/NAOMI_ChannelAware_5fold_1e-3_64')
CHANNELS_TO_REPORT = ('ScO2', 'MAP')  
COMPARE_WITH_DUAL = True        

MODEL_PREFIX = 'NAOMI_ChannelAware_Adapted'
EPS = 1e-8
N_FOLDS = 5
REGIONS = ['<1 min', '[1,10) min', '[10,20) min', '[20,30] min', 'ALL']
GAP_BINS = [0, 11, 119, 239, 360]
CHANNELS = {'ScO2': 'ScO2_only', 'MAP': 'MAP_only'}


def metrics(df):
    yt = df['y_true'].to_numpy(dtype=np.float64)
    yp = df['y_pred'].to_numpy(dtype=np.float64)
    good = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[good], yp[good]
    ae = np.abs(yp - yt)
    nz = np.abs(yt) > EPS
    return dict(
        mae=float(ae.mean()) if len(ae) else np.nan,
        mape_pct=float(100 * np.mean(ae[nz] / np.abs(yt[nz]))) if nz.any() else np.nan,
        n_points_total=int(len(ae)),
        n_points_used_mape=int(nz.sum()),
        n_points_excluded_gt0=int((~nz).sum()),
    )


def fmt(mean, sd):
    if pd.isna(mean):
        return 'NA'
    return f'{mean:.2f} ± {sd:.2f}' if pd.notna(sd) else f'{mean:.2f} ± NA'


def load_predictions(path, expected_target, expected_model=None):
    if not path.is_file():
        raise FileNotFoundError(f'Missing held-out test predictions: {path}\n'
                                'Finish the five-fold NAOMI training first.')
    df = pd.read_csv(path)
    needed = {'fold', 'target', 'y_true', 'y_pred', 'gap_len', 'segment_idx', 'time_idx'}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f'{path}: missing columns {sorted(missing)}')
    if 'split' in df.columns and not df['split'].astype(str).str.lower().eq('test').all():
        raise ValueError(f'{path}: contains non-test predictions')
    if 'model' not in df.columns:
        if expected_model is None:
            raise ValueError(f'{path}: missing model column')
        df['model'] = expected_model
    if expected_model is not None and set(df['model'].dropna().unique()) != {expected_model}:
        raise ValueError(f'{path}: expected model {expected_model!r}, found '
                         f'{df["model"].unique().tolist()}')
    # Explicit names only: a local index '0' is AMBIGUOUS for MAP-only inputs.
    df['target'] = df['target'].astype(str).replace({'ScO₂': 'ScO2'})
    if set(df['target'].unique()) != {expected_target}:
        raise ValueError(f'{path}: expected only target {expected_target!r}, found '
                         f'{df["target"].unique().tolist()}')
    for col in ['fold', 'gap_len', 'y_true', 'y_pred', 'segment_idx', 'time_idx']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    valid_idx = np.isfinite(df['fold']) & np.isfinite(df['segment_idx']) & np.isfinite(df['time_idx'])
    if not valid_idx.all():
        raise ValueError(f'{path}: invalid fold, segment_idx or time_idx')
    df = df.loc[np.isfinite(df['y_true']) & np.isfinite(df['y_pred'])].copy()
    if df.empty:
        raise ValueError(f'{path}: no finite held-out test predictions')
    if not np.all(df['fold'] == df['fold'].astype('int64')):
        raise ValueError(f'{path}: noninteger fold identifiers')
    df['fold'] = df['fold'].astype(int)
    if sorted(df['fold'].unique().tolist()) != list(range(1, N_FOLDS + 1)):
        raise ValueError(f'{path}: require all {N_FOLDS} folds before reporting; '
                         f'found {sorted(df["fold"].unique().tolist())}')
    keys = ['fold', 'segment_idx', 'time_idx', 'target']
    if df.duplicated(keys).any():
        raise ValueError(f'{path}: duplicate test prediction keys {keys}')
    return df


def add_gap_regions(df):
    out = df.copy()
    out['gap_region'] = pd.cut(out['gap_len'], bins=GAP_BINS,
                               labels=REGIONS[:-1], right=True, include_lowest=False)
    outside = int(out['gap_region'].isna().sum())
    if outside:
        print(f'WARNING: {outside:,} points outside defined gap bins; '
              'they remain included in ALL')
    return out


def expand_with_all(df):
    df = add_gap_regions(df)
    specific = df.loc[df['gap_region'].notna()].copy()
    allpoints = df.copy()
    allpoints['gap_region'] = 'ALL'
    result = pd.concat([specific, allpoints], ignore_index=True)
    result['gap_region'] = pd.Categorical(result['gap_region'].astype(str),
                                          categories=REGIONS, ordered=True)
    return result


def grouped(df, keys):
    rows = []
    for group_key, sub in df.groupby(keys, observed=True, sort=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        rows.append({**dict(zip(keys, group_key)), **metrics(sub)})
    result = pd.DataFrame(rows)
    if result.empty:
        raise ValueError(f'No groups available for {keys}')
    result['gap_region'] = pd.Categorical(result['gap_region'].astype(str),
                                          categories=REGIONS, ordered=True)
    return result.sort_values([c for c in ['target', 'model', 'gap_region', 'fold']
                               if c in keys]).reset_index(drop=True)


def create_reports(df):
    expanded = expand_with_all(df)
    foldwise = grouped(expanded, ['fold', 'target', 'model', 'gap_region'])
    summary = (foldwise.groupby(['target', 'model', 'gap_region'], observed=True)
               .agg(mae_mean=('mae', 'mean'), mae_std=('mae', 'std'),
                    mape_mean_pct=('mape_pct', 'mean'), mape_std_pct=('mape_pct', 'std'),
                    n_folds_mae=('mae', 'count'), n_folds_mape=('mape_pct', 'count'),
                    total_points=('n_points_total', 'sum'),
                    total_points_used_mape=('n_points_used_mape', 'sum'),
                    total_points_excluded_gt0=('n_points_excluded_gt0', 'sum'))
               .reset_index().sort_values(['target', 'model', 'gap_region']).reset_index(drop=True))
    pooled = grouped(expanded, ['target', 'model', 'gap_region'])
    paper_rows = []
    for (target, model), sub in summary.groupby(['target', 'model'], observed=True):
        row = {'target': target, 'model': model}
        for region in REGIONS:
            a = sub.loc[sub['gap_region'] == region]
            row[f'{region} | MAE'] = fmt(a.iloc[0]['mae_mean'], a.iloc[0]['mae_std']) if len(a) else 'NA'
            row[f'{region} | MAPE(%)'] = (fmt(a.iloc[0]['mape_mean_pct'], a.iloc[0]['mape_std_pct'])
                                          if len(a) else 'NA')
        paper_rows.append(row)
    return dict(foldwise=foldwise, summary=summary, pooled=pooled,
                paper=pd.DataFrame(paper_rows))


def write_reports(path, reports):
    path.mkdir(parents=True, exist_ok=True)
    names = {
        'foldwise': 'cv5_fold_mae_mape_by_gap_region.csv',
        'summary': 'cv5_summary_mean_std_mae_mape_by_gap_region.csv',
        'pooled': 'cv5_pooled_test_mae_mape_by_gap_region.csv',
        'paper': 'naomi_paper4_mae_mape_table.csv',
    }
    for key, filename in names.items():
        reports[key].to_csv(path / filename, index=False)
        print('Saved:', path / filename)


def paired_dual_comparison(single, dual_dir):
    """Paired scoring on identical held-out gap points; fail on unmatched keys."""
    dual_path = dual_dir / 'cv5_test_predictions_pooled.csv'
    if not dual_path.is_file():
        print('Dual-channel predictions not found; skipping paired ablation comparison.')
        return
    dual_all = pd.read_csv(dual_path)
    dual_name = MODEL_PREFIX
    required = {'fold', 'segment_idx', 'time_idx', 'target', 'gap_len', 'y_true', 'y_pred'}
    if required - set(dual_all.columns):
        raise ValueError(f'Dual CSV missing columns {sorted(required - set(dual_all.columns))}')
    if 'split' in dual_all.columns and not dual_all['split'].astype(str).str.lower().eq('test').all():
        raise ValueError('Dual predictions contain non-test rows')
    if 'model' in dual_all.columns and set(dual_all['model'].dropna()) != {dual_name}:
        raise ValueError(f'Unexpected dual-channel model labels: {dual_all["model"].unique()}')
    dual_all['target'] = dual_all['target'].astype(str).replace({'ScO₂': 'ScO2'})
    for col in ['fold', 'gap_len', 'y_true', 'y_pred', 'segment_idx', 'time_idx']:
        dual_all[col] = pd.to_numeric(dual_all[col], errors='coerce')
    if not np.isfinite(dual_all[['fold', 'segment_idx', 'time_idx']].to_numpy()).all():
        raise ValueError('Dual predictions have invalid fold/segment/time indices')
    dual_all = dual_all.loc[np.isfinite(dual_all['y_true']) & np.isfinite(dual_all['y_pred'])].copy()
    dual_all['fold'] = dual_all['fold'].astype(int)
    joined = []
    keys = ['fold', 'segment_idx', 'time_idx', 'target']
    for target, sdf in single.groupby('target', sort=False):
        ddf = dual_all.loc[dual_all['target'] == target].copy()
        if ddf.empty or sorted(ddf['fold'].unique().tolist()) != list(range(1, N_FOLDS + 1)):
            raise ValueError(f'Dual predictions target {target} missing a fold')
        if ddf.duplicated(keys).any():
            raise ValueError(f'Dual predictions target {target} have duplicate keys')
        subset_s = sdf[keys + ['y_true', 'y_pred', 'gap_len']].rename(
            columns={'y_true': 'y_true_single', 'y_pred': 'y_pred_single', 'gap_len': 'gap_len_single'})
        subset_d = ddf[keys + ['y_true', 'y_pred', 'gap_len']].rename(
            columns={'y_true': 'y_true_dual', 'y_pred': 'y_pred_dual', 'gap_len': 'gap_len_dual'})
        merged = subset_s.merge(subset_d, on=keys, validate='one_to_one', how='outer', indicator=True)
        if not merged['_merge'].eq('both').all():
            counts = merged['_merge'].value_counts().to_dict()
            raise ValueError(f'Single/dual held-out points do not match for {target}: {counts}')
        if not np.allclose(merged['y_true_single'], merged['y_true_dual'], atol=1e-5, rtol=1e-6):
            raise ValueError(f'Single/dual ground truth differs for {target}')
        if not np.array_equal(merged['gap_len_single'].to_numpy(),
                              merged['gap_len_dual'].to_numpy()):
            raise ValueError(f'Single/dual gap lengths differ for {target}')
        merged = merged.rename(columns={'y_true_single': 'y_true', 'gap_len_single': 'gap_len'})
        joined.append(merged)
    paired = pd.concat(joined, ignore_index=True)
    expanded = expand_with_all(paired)
    records = []
    for (fold, target, region), part in expanded.groupby(
            ['fold', 'target', 'gap_region'], observed=True, sort=False):
        one = metrics(part.assign(y_pred=part['y_pred_single']))
        two = metrics(part.assign(y_pred=part['y_pred_dual']))
        records.append(dict(
            fold=int(fold), target=target, gap_region=region,
            single_mae=one['mae'], dual_mae=two['mae'],
            mae_reduction=one['mae']-two['mae'],
            mae_reduction_pct=(100*(one['mae']-two['mae'])/one['mae']
                               if one['mae'] > EPS else np.nan),
            single_mape_pct=one['mape_pct'], dual_mape_pct=two['mape_pct'],
            mape_reduction_pct=(100*(one['mape_pct']-two['mape_pct'])/one['mape_pct']
                                if pd.notna(one['mape_pct']) and one['mape_pct'] > EPS else np.nan),
            n_points=one['n_points_total']))
    foldwise = pd.DataFrame(records)
    foldwise['gap_region'] = pd.Categorical(foldwise['gap_region'].astype(str),
                                            categories=REGIONS, ordered=True)
    foldwise = foldwise.sort_values(['target','gap_region','fold']).reset_index(drop=True)
    summary = (foldwise.groupby(['target','gap_region'], observed=True)
               .agg(single_mae_mean=('single_mae','mean'),single_mae_std=('single_mae','std'),
                    dual_mae_mean=('dual_mae','mean'),dual_mae_std=('dual_mae','std'),
                    mae_reduction_mean=('mae_reduction','mean'),
                    mae_reduction_std=('mae_reduction','std'),
                    mae_reduction_pct_mean=('mae_reduction_pct','mean'),
                    mae_reduction_pct_std=('mae_reduction_pct','std'),
                    single_mape_mean_pct=('single_mape_pct','mean'),
                    single_mape_std_pct=('single_mape_pct','std'),
                    dual_mape_mean_pct=('dual_mape_pct','mean'),
                    dual_mape_std_pct=('dual_mape_pct','std'),
                    n_folds=('fold','nunique'), total_points=('n_points','sum'))
               .reset_index().sort_values(['target','gap_region']).reset_index(drop=True))
    paper_rows = []
    for target, sub in summary.groupby('target'):
        for region in REGIONS:
            match = sub.loc[sub['gap_region'] == region]
            if match.empty:
                continue
            r = match.iloc[0]
            paper_rows.append(dict(target=target, gap_region=region,
                 single_MAE=fmt(r['single_mae_mean'],r['single_mae_std']),
                 dual_MAE=fmt(r['dual_mae_mean'],r['dual_mae_std']),
                 dual_MAE_improvement_pct=fmt(r['mae_reduction_pct_mean'], r['mae_reduction_pct_std']),
                 single_MAPE_pct=fmt(r['single_mape_mean_pct'],r['single_mape_std_pct']),
                 dual_MAPE_pct=fmt(r['dual_mape_mean_pct'],r['dual_mape_std_pct'])))
    single_root = OUT_ROOT
    single_root.mkdir(parents=True, exist_ok=True)
    for filename, table in (
        ('cv5_single_vs_dual_foldwise_by_gap.csv',foldwise),
        ('cv5_single_vs_dual_mean_std_by_gap.csv',summary),
        ('cv5_single_vs_dual_paper_table.csv',pd.DataFrame(paper_rows)),
    ):
        table.to_csv(single_root / filename, index=False)
        print('Saved:', single_root / filename)
    print('\nPAIRED SINGLE vs DUAL FIVE-FOLD TEST MAE/MAPE:\n',
          pd.DataFrame(paper_rows).to_string(index=False))


def main():
    if not CHANNELS_TO_REPORT or any(c not in CHANNELS for c in CHANNELS_TO_REPORT):
        raise ValueError('CHANNELS_TO_REPORT must include ScO2 and/or MAP')
    all_reports, single_predictions = [], []
    for channel in CHANNELS_TO_REPORT:
        folder = OUT_ROOT / CHANNELS[channel]
        expected_model = f'{MODEL_PREFIX}_{channel}_only'
        pred = load_predictions(folder / 'cv5_test_predictions_pooled.csv',
                                expected_target=channel, expected_model=expected_model)
        reports = create_reports(pred)
        write_reports(folder, reports)
        all_reports.append(reports)
        single_predictions.append(pred)
        print(f'\n{channel}-ONLY FIVE-FOLD TEST MAE/MAPE:\n',
              reports['summary'].to_string(index=False))
        print(f'\n{channel}-ONLY MANUSCRIPT TABLE:\n',
              reports['paper'].to_string(index=False))
    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    root_names = {'foldwise':'cv5_single_channel_foldwise_mae_mape.csv',
                  'summary':'cv5_single_channel_summary_mean_std_mae_mape.csv',
                  'pooled':'cv5_single_channel_pooled_mae_mape.csv',
                  'paper':'naomi_single_channel_paper4_mae_mape_table.csv'}
    for key, filename in root_names.items():
        frame = pd.concat([x[key] for x in all_reports], ignore_index=True)
        frame.to_csv(OUT_ROOT / filename, index=False)
        print('Saved combined:', OUT_ROOT / filename)
    if COMPARE_WITH_DUAL:
        paired_dual_comparison(pd.concat(single_predictions, ignore_index=True), DUAL_DIR)
    print('\nDone. Reporting only; no model retraining or checkpoint changes.')


if __name__ == '__main__':
    main()


Saved: /content/drive/MyDrive/aim3/NAOMI_SingleChannel_5fold_1e-3_64/ScO2_only/cv5_fold_mae_mape_by_gap_region.csv
Saved: /content/drive/MyDrive/aim3/NAOMI_SingleChannel_5fold_1e-3_64/ScO2_only/cv5_summary_mean_std_mae_mape_by_gap_region.csv
Saved: /content/drive/MyDrive/aim3/NAOMI_SingleChannel_5fold_1e-3_64/ScO2_only/cv5_pooled_test_mae_mape_by_gap_region.csv
Saved: /content/drive/MyDrive/aim3/NAOMI_SingleChannel_5fold_1e-3_64/ScO2_only/naomi_paper4_mae_mape_table.csv

ScO2-ONLY FIVE-FOLD TEST MAE/MAPE:
 target                                model  gap_region  mae_mean  mae_std  mape_mean_pct  mape_std_pct  n_folds_mae  n_folds_mape  total_points  total_points_used_mape  total_points_excluded_gt0
  ScO2 NAOMI_ChannelAware_Adapted_ScO2_only      <1 min  0.681808 0.119318       1.164654      0.224388            5             5         25536                   25521                         15
  ScO2 NAOMI_ChannelAware_Adapted_ScO2_only  [1,10) min  1.028746 0.116154       1.832116      0

### T-test on patient-level comparison

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests

ROOT = Path(
    "/content/drive/MyDrive/aim3/NAOMI_SingleChannel_5fold_1e-3_64"
)

DUAL = Path(
    "/content/drive/MyDrive/aim3/NAOMI_ChannelAware_5fold_1e-3_64"
)

KEYS = [
    "fold", "patient_id", "segment_idx",
    "time_idx", "target", "gap_len"
]

REGIONS = [
    "<1 min",
    "[1,10) min",
    "[10,20) min",
    "[20,30] min"
]

rows = []

for target, folder in [
    ("ScO2", "ScO2_only"),
    ("MAP", "MAP_only")
]:

    print(f"\nProcessing {target}...")

    single = pd.read_csv(
        ROOT / folder / "cv5_test_predictions_pooled.csv"
    )

    dual = pd.read_csv(
        DUAL / "cv5_test_predictions_pooled.csv"
    )

    single = single[single["target"] == target].copy()
    dual = dual[dual["target"] == target].copy()

    if "split" in single.columns:
        single = single[single["split"] == "test"]

    if "split" in dual.columns:
        dual = dual[dual["split"] == "test"]

    for name, frame in [
        ("single", single),
        ("dual", dual)
    ]:
        if frame.duplicated(KEYS).any():
            raise ValueError(
                f"{target}: Duplicate {name} predictions"
            )


    paired = single[
        KEYS + ["y_true", "y_pred"]
    ].merge(
        dual[
            KEYS + ["y_true", "y_pred"]
        ],
        on=KEYS,
        how="outer",
        suffixes=("_single", "_dual"),
        indicator=True,
        validate="one_to_one"
    )

    if not paired["_merge"].eq("both").all():
        raise ValueError(
            f"{target}: Unmatched test points"
        )

    if not np.allclose(
        paired["y_true_single"],
        paired["y_true_dual"],
        equal_nan=True
    ):
        raise ValueError(
            f"{target}: Ground truths differ"
        )

    paired = paired[
        np.isfinite(paired["y_true_single"])
        & np.isfinite(paired["y_pred_single"])
        & np.isfinite(paired["y_pred_dual"])
    ].copy()

    yt = paired["y_true_single"]


    paired["AE_single"] = np.abs(
        paired["y_pred_single"] - yt
    )

    paired["AE_dual"] = np.abs(
        paired["y_pred_dual"] - yt
    )

    paired["delta_AE"] = (
        paired["AE_single"] - paired["AE_dual"]
    )

    paired["gap_region"] = pd.cut(
        paired["gap_len"],
        bins=[0, 11, 119, 239, 360],
        labels=REGIONS
    )


    for region in ["ALL"] + REGIONS:

        sub = (
            paired
            if region == "ALL"
            else paired[
                paired["gap_region"] == region
            ]
        )

        if sub.empty:
            continue

        fold_mae = sub.groupby("fold").agg(
            single_MAE=("AE_single", "mean"),
            dual_MAE=("AE_dual", "mean")
        )

        n_folds = len(fold_mae)

        if n_folds < 2:
            continue

        single_mean = fold_mae["single_MAE"].mean()
        single_std = fold_mae["single_MAE"].std(ddof=1)

        dual_mean = fold_mae["dual_MAE"].mean()
        dual_std = fold_mae["dual_MAE"].std(ddof=1)
        single_pooled_mae = sub["AE_single"].mean()
        dual_pooled_mae = sub["AE_dual"].mean()
        patient_delta = (
            sub.groupby("patient_id")["delta_AE"]
            .mean()
            .to_numpy()
        )

        n_patients = len(patient_delta)

        if n_patients < 2:
            continue

        t_stat, p_value = stats.ttest_1samp(
            patient_delta,
            popmean=0
        )

        mean_delta = patient_delta.mean()

        sem = stats.sem(
            patient_delta,
            ddof=1
        )

        critical = stats.t.ppf(
            0.975,
            df=n_patients - 1
        )

        ci_lower = mean_delta - critical * sem
        ci_upper = mean_delta + critical * sem
        rows.append({
            "target": target,
            "gap_region": region,

            "n_folds": n_folds,
            "n_patients": n_patients,
            "n_points": len(sub),

            "single_MAE": single_mean,
            "single_STD": single_std,

            "dual_MAE": dual_mean,
            "dual_STD": dual_std,

            "single_pooled_MAE": single_pooled_mae,
            "dual_pooled_MAE": dual_pooled_mae,

            "mean_patient_delta": mean_delta,

            "95CI_lower": ci_lower,
            "95CI_upper": ci_upper,

            "t_statistic": t_stat,
            "p_value": p_value
        })

results = pd.DataFrame(rows)

if results.empty:
    raise ValueError("No valid comparison results were generated.")

results["p_FDR"] = multipletests(
    results["p_value"].fillna(1),
    method="fdr_bh"
)[1]



display_results = results.copy()

display_results["Single MAE ± STD"] = results.apply(
    lambda r: (
        f"{r['single_MAE']:.2f} ± "
        f"{r['single_STD']:.2f}"
    ),
    axis=1
)

display_results["Dual MAE ± STD"] = results.apply(
    lambda r: (
        f"{r['dual_MAE']:.2f} ± "
        f"{r['dual_STD']:.2f}"
    ),
    axis=1
)

display_results["Mean difference"] = (
    results["mean_patient_delta"].map(
        lambda x: f"{x:.2f}"
    )
)

display_results["95% CI"] = results.apply(
    lambda r: (
        f"[{r['95CI_lower']:.2f}, "
        f"{r['95CI_upper']:.2f}]"
    ),
    axis=1
)

display_results["p_value"] = (
    results["p_value"].map(
        lambda x: f"{x:.4f}"
    )
)

display_results["p_FDR"] = (
    results["p_FDR"].map(
        lambda x: f"{x:.4f}"
    )
)

display_results = display_results[
    [
        "target",
        "gap_region",
        "n_folds",
        "n_patients",
        "n_points",
        "Single MAE ± STD",
        "Dual MAE ± STD",
        "Mean difference",
        "95% CI",
        "p_value",
        "p_FDR"
    ]
]


print("\n" + "=" * 100)
print("NAOMI SINGLE-CHANNEL VS DUAL-CHANNEL")
print("Five-fold MAE ± STD and patient-level paired t-tests")
print("=" * 100)

print(
    display_results.to_string(index=False)
)


results.to_csv(
    ROOT / "naomi_single_vs_dual_paired_ttest.csv",
    index=False
)


display_results.to_csv(
    ROOT / "naomi_single_vs_dual_paired_ttest_formatted.csv",
    index=False
)

print("\nResults saved successfully.")


Processing ScO2...

Processing MAP...

NAOMI SINGLE-CHANNEL VS DUAL-CHANNEL
Five-fold MAE ± STD and patient-level paired t-tests
target  gap_region  n_folds  n_patients  n_points Single MAE ± STD Dual MAE ± STD Mean difference         95% CI p_value  p_FDR
  ScO2         ALL        5          64    105364      1.18 ± 0.15    1.24 ± 0.15           -0.06 [-0.09, -0.03]  0.0001 0.0004
  ScO2      <1 min        5          64     25536      0.68 ± 0.12    0.71 ± 0.11           -0.03 [-0.04, -0.02]  0.0000 0.0000
  ScO2  [1,10) min        5          63     32896      1.03 ± 0.12    1.12 ± 0.10           -0.09 [-0.13, -0.05]  0.0000 0.0001
  ScO2 [10,20) min        5          50     23537      1.43 ± 0.29    1.51 ± 0.30           -0.06 [-0.12, -0.00]  0.0448 0.1000
  ScO2 [20,30] min        5          34     23395      1.71 ± 0.34    1.74 ± 0.39           -0.02  [-0.10, 0.05]  0.4949 0.4949
   MAP         ALL        5          64     39064      1.55 ± 0.23    1.56 ± 0.22           -0.04  [-0